In [1]:
from ortools.linear_solver import pywraplp
import numpy as np
import optuna

def solve_cable_optimization_or_tools():
    """
    Решение задачи оптимизации скрутки кабеля с использованием OR-Tools.
    """
    # Исходные данные
    white1 = 10700  # Белая жила №1
    white2 = 10300  # Белая жила №2
    
    black_pieces = [1859, 2800, 2596, 4600, 9100, 1600]
    n_pieces = len(black_pieces)
    
    # Создаем solver (целочисленное программирование)
    solver = pywraplp.Solver.CreateSolver('SCIP')
    if not solver:
        print("SCIP solver не доступен, пробуем CBC...")
        solver = pywraplp.Solver.CreateSolver('CBC')
    if not solver:
        print("Не удалось создать solver. Установите ortools правильно.")
        return 0
    
    # Переменные: x[i][j] - кусок i назначен в пару j (0 или 1)
    # j=0 для пары с white1, j=1 для пары с white2
    x = {}
    for i in range(n_pieces):
        for j in range(2):
            x[i, j] = solver.IntVar(0, 1, f'x_{i}_{j}')
    
    # Переменные для длины каждой пары
    pair_length = [solver.NumVar(0, max(white1, white2), f'pair_{j}') for j in range(2)]
    
    # Переменная для итоговой длины кабеля
    final_length = solver.NumVar(0, min(white1, white2), 'final_length')
    
    # Ограничения
    # 1. Каждый кусок черной жилы можно использовать только один раз
    for i in range(n_pieces):
        solver.Add(solver.Sum([x[i, j] for j in range(2)]) <= 1)
    
    # 2. Сумма черных кусков в каждой паре не должна превышать длину белой жилы
    for j in range(2):
        white_length = white1 if j == 0 else white2
        solver.Add(
            solver.Sum([black_pieces[i] * x[i, j] for i in range(n_pieces)]) <= white_length
        )
    
    # 3. Длина пары равна сумме черных кусков в ней
    for j in range(2):
        solver.Add(
            pair_length[j] == solver.Sum([black_pieces[i] * x[i, j] for i in range(n_pieces)])
        )
    
    # 4. Итоговая длина кабеля равна минимальной из длин пар
    solver.Add(final_length <= pair_length[0])
    solver.Add(final_length <= pair_length[1])
    solver.Add(final_length <= white1)
    solver.Add(final_length <= white2)
    
    # Целевая функция: максимизировать итоговую длину
    solver.Maximize(final_length)
    
    # Решаем задачу
    status = solver.Solve()
    
    if status == pywraplp.Solver.OPTIMAL:
        print("\n" + "=" * 60)
        print("ОР-Tools решение (оптимальное)")
        print("=" * 60)
        
        final_len = final_length.solution_value()
        print(f"Максимальная длина готового кабеля: {final_len:.2f} м")
        
        # Собираем результаты по парам
        pair1_pieces = []
        pair2_pieces = []
        
        for i in range(n_pieces):
            for j in range(2):
                if x[i, j].solution_value() > 0.5:
                    if j == 0:
                        pair1_pieces.append(black_pieces[i])
                    else:
                        pair2_pieces.append(black_pieces[i])
        
        len_pair1 = sum(pair1_pieces)
        len_pair2 = sum(pair2_pieces)
        
        print(f"\nПара 1 (с белой №1 {white1} м):")
        print(f"  Черные куски: {pair1_pieces}")
        print(f"  Сумма черных: {len_pair1:.2f} м")
        print(f"  Остаток белой №1: {white1 - final_len:.2f} м")
        
        print(f"\nПара 2 (с белой №2 {white2} м):")
        print(f"  Черные куски: {pair2_pieces}")
        print(f"  Сумма черных: {len_pair2:.2f} м")
        print(f"  Остаток белой №2: {white2 - final_len:.2f} м")
        
        unused = [p for p in black_pieces if p not in pair1_pieces + pair2_pieces]
        if unused:
            print(f"\nНеиспользованные куски черной: {unused}")
        
        total_black_used = len_pair1 + len_pair2
        print(f"\nВсего использовано черной: {total_black_used:.2f} м")
        print(f"Коэффициент использования черной: {total_black_used/sum(black_pieces)*100:.1f}%")
        return final_len
    else:
        print("Оптимальное решение не найдено!")
        return 0


def solve_cable_optimization_optuna(n_trials=10000):
    """
    Решение задачи оптимизации с использованием Optuna.
    """
    black_pieces = [1859, 2800, 2596, 4600, 9100, 1600]
    white1, white2 = 10700, 10300
    
    def objective(trial):
        assignments = []
        for i in range(len(black_pieces)):
            assignment = trial.suggest_categorical(f'piece_{i}', [0, 1, 2])
            assignments.append(assignment)
        
        pair1_sum = sum(black_pieces[i] for i, a in enumerate(assignments) if a == 1)
        pair2_sum = sum(black_pieces[i] for i, a in enumerate(assignments) if a == 2)
        
        if pair1_sum > white1 or pair2_sum > white2:
            return -1.0  # штраф
        
        final_length = min(pair1_sum, pair2_sum, white1, white2)
        return final_length
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    print("\n" + "=" * 60)
    print("Optuna решение (лучшее найденное)")
    print("=" * 60)
    
    best_value = study.best_value
    print(f"Максимальная длина готового кабеля: {best_value:.2f} м")
    
    best_params = study.best_params
    pair1_pieces = []
    pair2_pieces = []
    
    for i in range(len(black_pieces)):
        assignment = best_params[f'piece_{i}']
        if assignment == 1:
            pair1_pieces.append(black_pieces[i])
        elif assignment == 2:
            pair2_pieces.append(black_pieces[i])
    
    len_pair1 = sum(pair1_pieces)
    len_pair2 = sum(pair2_pieces)
    
    print(f"\nПара 1 (с белой №1): {pair1_pieces} сумма = {len_pair1}")
    print(f"Пара 2 (с белой №2): {pair2_pieces} сумма = {len_pair2}")
    
    unused = [p for p in black_pieces if p not in pair1_pieces + pair2_pieces]
    if unused:
        print(f"Неиспользованные куски: {unused}")
    
    return best_value


def brute_force_check():
    """
    Полный перебор (для верификации).
    """
    print("\n" + "=" * 60)
    print("Проверка полным перебором")
    print("=" * 60)
    
    black_pieces = [1859, 2800, 2596, 4600, 9100, 1600]
    white1, white2 = 10700, 10300
    n = len(black_pieces)
    
    best_length = 0
    best_combo = None
    
    # 3^n комбинаций (0 - не используется, 1 - в пару1, 2 - в пару2)
    for mask in range(3**n):
        assignment = []
        temp = mask
        for _ in range(n):
            assignment.append(temp % 3)
            temp //= 3
        
        pair1_sum = sum(black_pieces[i] for i, a in enumerate(assignment) if a == 1)
        pair2_sum = sum(black_pieces[i] for i, a in enumerate(assignment) if a == 2)
        
        if pair1_sum <= white1 and pair2_sum <= white2:
            current_length = min(pair1_sum, pair2_sum, white1, white2)
            if current_length > best_length:
                best_length = current_length
                best_combo = assignment.copy()
    
    print(f"Максимальная длина (полный перебор): {best_length:.2f} м")
    
    if best_combo:
        pair1 = [black_pieces[i] for i, a in enumerate(best_combo) if a == 1]
        pair2 = [black_pieces[i] for i, a in enumerate(best_combo) if a == 2]
        print(f"Пара 1: {pair1}, сумма = {sum(pair1)}")
        print(f"Пара 2: {pair2}, сумма = {sum(pair2)}")
    
    return best_length


if __name__ == "__main__":
    # Выводим условие задачи
    print("Задача: оптимизация скрутки кабеля 2х(2х0.75)")
    print("Белая №1: 10700 м")
    print("Белая №2: 10300 м")
    print("Черная: 1859, 2800, 2596, 4600, 9100, 1600 м")
    print()
    
    # Полный перебор (проверка)
    brute_length = brute_force_check()
    
    # OR-Tools
    or_length = solve_cable_optimization_or_tools()
    
    # Optuna
    optuna_length = solve_cable_optimization_optuna(n_trials=5000)
    
    print("\n" + "=" * 60)
    print("Сводка результатов")
    print("=" * 60)
    print(f"Полный перебор: {brute_length:.2f} м")
    print(f"OR-Tools:       {or_length:.2f} м")
    print(f"Optuna:         {optuna_length:.2f} м")

c:\Users\user\Desktop\Projects\lenght\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-02-27 20:46:22,486] A new study created in memory with name: no-name-1983750b-2958-4284-9ff8-7ec276d9afd6


Задача: оптимизация скрутки кабеля 2х(2х0.75)
Белая №1: 10700 м
Белая №2: 10300 м
Черная: 1859, 2800, 2596, 4600, 9100, 1600 м


Проверка полным перебором
Максимальная длина (полный перебор): 9996.00 м
Пара 1: [9100, 1600], сумма = 10700
Пара 2: [2800, 2596, 4600], сумма = 9996

ОР-Tools решение (оптимальное)
Максимальная длина готового кабеля: 9996.00 м

Пара 1 (с белой №1 10700 м):
  Черные куски: [9100, 1600]
  Сумма черных: 10700.00 м
  Остаток белой №1: 704.00 м

Пара 2 (с белой №2 10300 м):
  Черные куски: [2800, 2596, 4600]
  Сумма черных: 9996.00 м
  Остаток белой №2: 304.00 м

Неиспользованные куски черной: [1859]

Всего использовано черной: 20696.00 м
Коэффициент использования черной: 91.8%


Best trial: 0. Best value: 4400:   0%|          | 2/5000 [00:00<08:38,  9.65it/s]

[I 2026-02-27 20:46:22,630] Trial 0 finished with value: 4400.0 and parameters: {'piece_0': 0, 'piece_1': 2, 'piece_2': 1, 'piece_3': 1, 'piece_4': 0, 'piece_5': 2}. Best is trial 0 with value: 4400.0.
[I 2026-02-27 20:46:22,637] Trial 1 finished with value: -1.0 and parameters: {'piece_0': 2, 'piece_1': 1, 'piece_2': 1, 'piece_3': 2, 'piece_4': 2, 'piece_5': 0}. Best is trial 0 with value: 4400.0.
[I 2026-02-27 20:46:22,641] Trial 2 finished with value: 2596.0 and parameters: {'piece_0': 0, 'piece_1': 0, 'piece_2': 1, 'piece_3': 0, 'piece_4': 2, 'piece_5': 0}. Best is trial 0 with value: 4400.0.


Best trial: 7. Best value: 4455:   0%|          | 21/5000 [00:00<01:05, 76.29it/s]

...

Best trial: 45. Best value: 9100:  49%|████▉     | 2460/5000 [01:11<02:10, 19.51it/s]

[I 2026-02-27 20:47:34,259] Trial 2457 finished with value: -1.0 and parameters: {'piece_0': 1, 'piece_1': 1, 'piece_2': 1, 'piece_3': 1, 'piece_4': 2, 'piece_5': 0}. Best is trial 45 with value: 9100.0.
[I 2026-02-27 20:47:34,309] Trial 2458 finished with value: -1.0 and parameters: {'piece_0': 0, 'piece_1': 1, 'piece_2': 1, 'piece_3': 2, 'piece_4': 2, 'piece_5': 0}. Best is trial 45 with value: 9100.0.
[I 2026-02-27 20:47:34,360] Trial 2459 finished with value: 9100.0 and parameters: {'piece_0': 0, 'piece_1': 1, 'piece_2': 1, 'piece_3': 1, 'piece_4': 2, 'piece_5': 0}. Best is trial 45 with value: 9100.0.
[I 2026-02-27 20:47:34,405] Trial 2460 finished with value: 0.0 and parameters: {'piece_0': 0, 'piece_1': 1, 'piece_2': 1, 'piece_3': 1, 'piece_4': 0, 'piece_5': 0}. Best is trial 45 with value: 9100.0.


Best trial: 45. Best value: 9100:  49%|████▉     | 2465/5000 [01:12<01:14, 34.19it/s]


[I 2026-02-27 20:47:34,450] Trial 2461 finished with value: -1.0 and parameters: {'piece_0': 0, 'piece_1': 1, 'piece_2': 2, 'piece_3': 1, 'piece_4': 2, 'piece_5': 0}. Best is trial 45 with value: 9100.0.
[I 2026-02-27 20:47:34,495] Trial 2462 finished with value: 9100.0 and parameters: {'piece_0': 0, 'piece_1': 1, 'piece_2': 1, 'piece_3': 1, 'piece_4': 2, 'piece_5': 0}. Best is trial 45 with value: 9100.0.
[I 2026-02-27 20:47:34,537] Trial 2463 finished with value: 9100.0 and parameters: {'piece_0': 0, 'piece_1': 1, 'piece_2': 1, 'piece_3': 1, 'piece_4': 2, 'piece_5': 0}. Best is trial 45 with value: 9100.0.
[I 2026-02-27 20:47:34,583] Trial 2464 finished with value: 9100.0 and parameters: {'piece_0': 0, 'piece_1': 1, 'piece_2': 1, 'piece_3': 1, 'piece_4': 2, 'piece_5': 0}. Best is trial 45 with value: 9100.0.
[W 2026-02-27 20:47:34,603] Trial 2465 failed with parameters: {} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\user\De

KeyboardInterrupt: 